
# KDMS V4 — 문헌 근거·사용감·피부유형 조합 Feature Engineering

이 노트북은 V3 기능군 Count를 기본으로 유지하면서 다음 V4 블록을 추가합니다.

| 구분 | 핵심 피처 예시 | 역할 | 주요 근거 |
|---|---|---|---|
| **① V3 기본** | 보습·유연·밀폐·장벽·진정 등 Count 12개 | 제품의 기본 기능성 구성 | 현재 내부 Ablation에서 가장 안정적인 기본 Feature |
| **② 건성·약건성 조합** | `건성×보습×밀폐`, `건성×세라마이드×콜레스테롤×지방산` | 수분 공급·장벽지질 완성도 | Vaillant et al., 2020, PMID 31532576; Man et al., 1996, PMID 8618046 |
| **③ 지성·트러블 조합** | `지성×나이아신아마이드`, `트러블×나이아신아마이드×세라마이드` | 피지 및 여드름 관리 중 장벽 보조 후보 | Draelos et al., 2006, PMID 16766489; Tempark et al., 2024, PMID 38299457 |
| **④ 민감성 주의 조합** | `민감성×향료 알레르겐`, `민감성×산·각질`, `민감성×에센셜오일` | 불만족·자극 연관 후보 | EU SCCS fragrance allergen opinions; Johansen, 2003, PMID 14572300 |
| **⑤ 사용감 처방군** | 가벼운 에스터, 휘발성/비휘발성 실리콘, 식물성 오일, 지방알코올, 왁스·버터 | 끈적임·유분감·퍼짐성·잔여감 Proxy | Parente et al., 2005, PMID 16116522; Lukic et al., 2012, PMID 22085371 |
| **⑥ 피부×사용감** | `지성×heavy_load`, `지성×light_system`, `건성×occlusive_support` | 피부유형과 처방 감각의 적합성 학습 | 사용감 연구로 후보만 선정하고 실제 방향은 모델이 학습 |
| **⑦ 문헌 집계점수** | `support_score`, `caution_score`, `conflict_score` | 여러 규칙을 근거 신뢰도로 압축 | 가중치는 임상 효과 크기가 아니라 근거 신뢰도 |
| **⑧ Fold 내부 posterior** | `posterior_net`, `posterior_positive`, `posterior_negative` | 문헌 prior를 실제 Train 리뷰로 보정 | Cross-fitted empirical-Bayes target encoding 원칙 |

## 논문·공식 출처

- [Combined effects of glycerol and petrolatum — PMID 31532576](https://pubmed.ncbi.nlm.nih.gov/31532576/)
- [Optimization of physiological lipid mixtures — PMID 8618046](https://pubmed.ncbi.nlm.nih.gov/8618046/)
- [The effect of 2% niacinamide on facial sebum production — PMID 16766489](https://pubmed.ncbi.nlm.nih.gov/16766489/)
- [Ceramides and niacinamide moisturizer in acne — PMID 38299457](https://pubmed.ncbi.nlm.nih.gov/38299457/)
- [Study of sensory properties of emollients — PMID 16116522](https://pubmed.ncbi.nlm.nih.gov/16116522/)
- [Influence of emollient on sensory properties — PMID 22085371](https://pubmed.ncbi.nlm.nih.gov/22085371/)
- [EU Scientific Committee on Consumer Safety](https://health.ec.europa.eu/scientific-committees/scientific-committee-consumer-safety-sccs_en)
- [Scikit-learn TargetEncoder: cross fitting and empirical-Bayes smoothing](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.TargetEncoder.html)

> 본 연구의 V4는 임상적 효능·안전성을 판정하지 않습니다.  
> 논문은 조합 후보와 근거 신뢰도에만 사용하며, 온라인 리뷰 위험 방향은 모델이 다시 학습합니다.


## 1. Google Drive 연결 및 실행 설정

In [ ]:

from pathlib import Path
from collections import defaultdict
from itertools import combinations
import hashlib
import json
import math
import re
import unicodedata
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

try:
    from catboost import CatBoostClassifier
except ImportError:
    !pip -q install catboost
    from catboost import CatBoostClassifier

# ---------------------------------------------------------
# Colab에서는 Google Drive, 로컬 검증에서는 /mnt/data 사용
# ---------------------------------------------------------
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    BASE_DIR = Path(
        '/content/drive/MyDrive/0_ASAC_11기_ML_2조/haha'
    )

    LONG_PATH = (
        BASE_DIR
        / '2_전처리'
        / '전성분long'
        / '전성분_final.csv'
    )

    V3_PATH = (
        BASE_DIR
        / '3_feature'
        / '서영'
        / 'v3'
        / 'v3_function_features.csv'
    )

    OUTPUT_DIR = (
        BASE_DIR
        / '3_feature'
        / '서영'
        / '04'
    )
else:
    BASE_DIR = Path('/mnt/data')
    LONG_PATH = BASE_DIR / '전성분_final.csv'
    V3_PATH = BASE_DIR / 'v3_function_features.csv'
    OUTPUT_DIR = BASE_DIR / 'v4_final_local_test'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 리뷰 경로를 정확히 알면 직접 지정할 수 있습니다.
# None이면 파일명에 리뷰·감성·최종이 포함된 CSV를 자동 탐색합니다.
REVIEW_PATH = None

# Target
MIN_REVIEWS_PER_PRODUCT_SKIN = 5
HIGH_RISK_THRESHOLD = 0.0833

# 평가
N_SPLITS = 5
N_REPEATS = 2
INNER_POSTERIOR_SPLITS = 4
RANDOM_SEED = 2026

# 모델 학습 안정화
USE_REVIEW_COUNT_WEIGHT = True
MAX_REVIEW_WEIGHT_COUNT = 50

# 시간이 오래 걸리면 False로 변경
RUN_CATBOOST = True
RUN_ELASTICNET = False  # 계수 해석까지 할 때 True
RUN_RANDOM_RULE_CONTROL = True
N_RANDOM_RULE_SETS = 10  # 최종 논문용은 50~100 권장

print("전성분:", LONG_PATH)
print("V3:", V3_PATH)
print("출력:", OUTPUT_DIR)


## 2. 데이터 불러오기 및 리뷰 Target 생성

In [ ]:

def normalize_text(value):
    if pd.isna(value):
        return ''
    return unicodedata.normalize('NFKC', str(value)).strip()

def find_review_file(base_dir):
    candidates = []

    for path in base_dir.rglob('*.csv'):
        name = normalize_text(path.name)

        if all(token in name for token in ['리뷰', '감성', '최종']):
            candidates.append(path)

    if not candidates:
        raise FileNotFoundError(
            "리뷰 감성 최종 CSV를 찾지 못했습니다. "
            "REVIEW_PATH에 직접 경로를 입력하세요."
        )

    return sorted(
        candidates,
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )[0]

if REVIEW_PATH is None:
    REVIEW_PATH = find_review_file(BASE_DIR)
else:
    REVIEW_PATH = Path(REVIEW_PATH)

long_df = pd.read_csv(LONG_PATH, encoding='utf-8-sig')
v3_df = pd.read_csv(V3_PATH, encoding='utf-8-sig')
review_df = pd.read_csv(REVIEW_PATH, encoding='utf-8-sig')

required_long = {
    'product_id',
    'product_name',
    'canonical_name',
    'ingredient_order',
}
required_v3 = {'product_id', 'product_name'}
required_review = {'상품번호', '피부타입', '리뷰분류'}

for label, frame, required in [
    ('전성분', long_df, required_long),
    ('V3', v3_df, required_v3),
    ('리뷰', review_df, required_review),
]:
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"{label} 필수 컬럼 누락: {sorted(missing)}")

long_df['canonical_name'] = (
    long_df['canonical_name']
    .astype('string')
    .str.strip()
)
long_df['ingredient_order'] = pd.to_numeric(
    long_df['ingredient_order'],
    errors='coerce',
)

long_df = long_df[
    long_df['canonical_name'].notna()
    & long_df['canonical_name'].ne('')
].copy()

review_df['상품번호'] = review_df['상품번호'].astype(str)
v3_df['product_id'] = v3_df['product_id'].astype(str)
long_df['product_id'] = long_df['product_id'].astype(str)

review_df['is_negative'] = (
    review_df['리뷰분류']
    .astype(str)
    .str.strip()
    .eq('부정')
    .astype('int8')
)

target_df = (
    review_df
    .groupby(['상품번호', '피부타입'], as_index=False)
    .agg(
        review_count=('is_negative', 'size'),
        negative_count=('is_negative', 'sum'),
    )
)

target_df['negative_rate'] = (
    target_df['negative_count']
    / target_df['review_count']
)

target_df['high_risk'] = (
    target_df['negative_rate']
    >= HIGH_RISK_THRESHOLD
).astype('int8')

target_df = target_df[
    target_df['review_count']
    >= MIN_REVIEWS_PER_PRODUCT_SKIN
].copy()

print("리뷰 파일:", REVIEW_PATH)
print("전체 리뷰 수:", len(review_df))
print("제품×피부유형 Target 행:", len(target_df))
print("High-risk 비율:", f"{target_df['high_risk'].mean():.2%}")
display(target_df.head())



## 3. 제품 성분 집합·처방 그룹·성분 순서 Proxy

전성분 순서는 실제 함량이 아니라 **약한 prominence proxy**로만 사용합니다.

- `top10`: 상위 10번째 이내에 표기됐는지
- `prominence`: 제품 내 표기 순서가 앞쪽일수록 1에 가까움


In [ ]:

product_ingredient_sets = (
    long_df.groupby('product_id')['canonical_name']
    .apply(set)
    .to_dict()
)

def formula_hash(ingredients):
    text = '\x1f'.join(sorted(ingredients))
    return hashlib.md5(text.encode('utf-8')).hexdigest()

formula_group_map = {
    product_id: formula_hash(ingredients)
    for product_id, ingredients in product_ingredient_sets.items()
}

product_max_order = (
    long_df.groupby('product_id')['ingredient_order']
    .max()
    .to_dict()
)

def ingredient_order_feature(product_id, matcher):
    product_rows = long_df[
        long_df['product_id'].eq(product_id)
    ]

    matched = product_rows[
        product_rows['canonical_name'].map(matcher)
    ]

    if matched.empty:
        return {
            'has': 0,
            'top10': 0,
            'min_order': 0.0,
            'prominence': 0.0,
        }

    min_order = float(matched['ingredient_order'].min())
    max_order = float(
        product_max_order.get(product_id, min_order)
    )

    if max_order <= 1:
        prominence = 1.0
    else:
        prominence = 1.0 - (
            (min_order - 1.0) / (max_order - 1.0)
        )

    return {
        'has': 1,
        'top10': int(min_order <= 10),
        'min_order': min_order,
        'prominence': float(
            np.clip(prominence, 0.0, 1.0)
        ),
    }

product_ids = sorted(
    set(v3_df['product_id'])
    & set(long_df['product_id'])
)

order_rows = []

for product_id in product_ids:
    niacinamide = ingredient_order_feature(
        product_id,
        lambda name: name == '나이아신아마이드',
    )
    panthenol = ingredient_order_feature(
        product_id,
        lambda name: name == '판테놀',
    )
    ceramide = ingredient_order_feature(
        product_id,
        lambda name: str(name).startswith('세라마이드'),
    )
    salicylic = ingredient_order_feature(
        product_id,
        lambda name: (
            name == '살리실릭애씨드'
            or name == '카프릴로일살리실릭애씨드'
        ),
    )

    order_rows.append({
        'product_id': product_id,
        'v4_has_niacinamide': niacinamide['has'],
        'v4_niacinamide_top10': niacinamide['top10'],
        'v4_niacinamide_prominence': niacinamide['prominence'],
        'v4_has_panthenol': panthenol['has'],
        'v4_panthenol_top10': panthenol['top10'],
        'v4_panthenol_prominence': panthenol['prominence'],
        'v4_has_ceramide': ceramide['has'],
        'v4_ceramide_prominence': ceramide['prominence'],
        'v4_has_salicylic': salicylic['has'],
        'v4_salicylic_prominence': salicylic['prominence'],
        'formula_group': formula_group_map.get(product_id),
    })

order_df = pd.DataFrame(order_rows)
display(order_df.head())



## 4. 사용감 처방군 정의

다음 분류는 완제품의 실제 감각평가값이 아니라 전성분명 기반 Proxy입니다.

- Light ester
- Volatile silicone
- Nonvolatile silicone
- Hydrocarbon occlusive
- Fatty alcohol
- Plant oil
- Wax/butter
- Essential oil

한 성분이 둘 이상의 보조 분류에 포함될 수 있습니다.


In [ ]:

ESSENTIAL_OIL_KEYWORDS = [
    '라벤더',
    '베르가모트',
    '티트리',
    '로즈마리',
    '페퍼민트',
    '유칼립투스',
    '레몬',
    '라임',
    '오렌지',
    '자몽',
    '만다린',
    '일랑일랑',
    '클라리',
    '제라늄',
]

def classify_sensory(name):
    groups = set()
    value = str(name)

    # 가벼운 에스터·에터 계열
    if any(token in value for token in [
        '세틸에틸헥사노에이트',
        '다이카프릴릴카보네이트',
        '다이카프릴릴에터',
        '트라이에틸헥사노인',
        '아이소노닐아이소노나노에이트',
        '아이소프로필미리스테이트',
        'C12-15알킬벤조에이트',
        '코코-카프릴레이트/카프레이트',
    ]):
        groups.add('light_ester')

    # 휘발성 또는 빠른 퍼짐의 실리콘 계열
    if any(token in value for token in [
        '사이클로펜타실록세인',
        '사이클로헥사실록세인',
        '메틸트라이메티콘',
        '트라이실록세인',
        '카프릴릴메티콘',
    ]):
        groups.add('volatile_silicone')

    # 비휘발성 실리콘 계열
    if any(token in value for token in [
        '다이메티콘',
        '다이메티콘올',
        '비닐다이메티콘',
        '폴리메틸실세스퀴옥세인',
    ]):
        groups.add('nonvolatile_silicone')

    # 탄화수소형 밀폐·유연 성분
    if any(token in value for token in [
        '페트롤라툼',
        '미네랄오일',
        '파라핀',
        '하이드로제네이티드폴리데센',
        '하이드로제네이티드폴리아이소부텐',
    ]):
        groups.add('hydrocarbon_occlusive')

    # 지방알코올
    if any(token in value for token in [
        '세테아릴알코올',
        '세틸알코올',
        '스테아릴알코올',
        '베헤닐알코올',
        '아라키딜알코올',
    ]):
        groups.add('fatty_alcohol')

    # 에센셜오일을 먼저 분리
    is_essential_oil = (
        value.endswith('오일')
        and any(
            token in value
            for token in ESSENTIAL_OIL_KEYWORDS
        )
    )

    if is_essential_oil:
        groups.add('essential_oil')
    elif value.endswith('오일'):
        groups.add('plant_oil')

    if (
        value.endswith('버터')
        or '왁스' in value
        or value in {'세레신', '오조케라이트'}
    ):
        groups.add('wax_butter')

    return groups

sensory_long = (
    long_df[
        ['product_id', 'canonical_name']
    ]
    .drop_duplicates()
    .copy()
)

sensory_long['sensory_groups'] = (
    sensory_long['canonical_name']
    .map(classify_sensory)
)

sensory_long = sensory_long[
    sensory_long['sensory_groups'].map(bool)
].explode('sensory_groups')

SENSORY_GROUPS = [
    'light_ester',
    'volatile_silicone',
    'nonvolatile_silicone',
    'hydrocarbon_occlusive',
    'fatty_alcohol',
    'plant_oil',
    'wax_butter',
    'essential_oil',
]

sensory_wide = (
    sensory_long
    .groupby(
        ['product_id', 'sensory_groups']
    )['canonical_name']
    .nunique()
    .unstack(fill_value=0)
    .reindex(columns=SENSORY_GROUPS, fill_value=0)
)

sensory_wide.columns = [
    f'v4_sensory_count_{group}'
    for group in sensory_wide.columns
]

sensory_wide = sensory_wide.reset_index()

for group in SENSORY_GROUPS:
    column = f'v4_sensory_count_{group}'
    sensory_wide[column] = (
        sensory_wide[column]
        .fillna(0)
        .astype('int16')
    )

display(sensory_wide.head())


## 5. 제품 단위 V3·순서·사용감 결합

In [ ]:

product_feature_df = (
    v3_df
    .merge(
        order_df,
        on='product_id',
        how='inner',
        validate='one_to_one',
    )
    .merge(
        sensory_wide,
        on='product_id',
        how='left',
        validate='one_to_one',
    )
)

sensory_columns = [
    f'v4_sensory_count_{group}'
    for group in SENSORY_GROUPS
]

product_feature_df[sensory_columns] = (
    product_feature_df[sensory_columns]
    .fillna(0)
)

CERAMIDE_NAMES = {
    name
    for name in long_df['canonical_name'].unique()
    if str(name).startswith('세라마이드')
}
CHOLESTEROL_NAMES = {'콜레스테롤'}
FATTY_ACID_NAMES = {
    '스테아릭애씨드',
    '팔미틱애씨드',
    '리놀레익애씨드',
    '리놀레닉애씨드',
}
FRAGRANCE_ALLERGEN_NAMES = {
    '향료',
    '리모넨',
    '리날룰',
    '시트랄',
    '시트로넬올',
    '제라니올',
    '유제놀',
    '아이소유제놀',
    '쿠마린',
    '파네솔',
    '벤질살리실레이트',
    '벤질벤조에이트',
    '헥실신남알',
    '알파-아이소메틸아이오논',
    '아니스알코올',
}
ACID_EXFOLIANT_NAMES = {
    '살리실릭애씨드',
    '카프릴로일살리실릭애씨드',
    '글라이콜릭애씨드',
    '락틱애씨드',
    '만델릭애씨드',
    '글루코노락톤',
    '락토바이오닉애씨드',
    '아젤라익애씨드',
    '석시닉애씨드',
}

fact_rows = []

for product_id in product_feature_df['product_id']:
    ingredients = product_ingredient_sets.get(
        product_id,
        set(),
    )

    fact_rows.append({
        'product_id': product_id,
        'v4_has_cholesterol': int(
            bool(ingredients & CHOLESTEROL_NAMES)
        ),
        'v4_has_fatty_acid': int(
            bool(ingredients & FATTY_ACID_NAMES)
        ),
        'v4_fragrance_allergen_count': len(
            ingredients & FRAGRANCE_ALLERGEN_NAMES
        ),
        'v4_acid_exact_count': len(
            ingredients & ACID_EXFOLIANT_NAMES
        ),
    })

fact_df = pd.DataFrame(fact_rows)

product_feature_df = product_feature_df.merge(
    fact_df,
    on='product_id',
    how='left',
    validate='one_to_one',
)

display(product_feature_df.head())



## 6. 문헌 근거 규칙 정의

`evidence_weight`는 다음 곱으로 계산합니다.

```text
연구설계 신뢰도 × 피부유형 직접성 × 현재 크림 데이터 적용 적합성
```

이 값은 효능의 크기가 아닙니다.


In [ ]:

RULE_DEFINITIONS = [
    {
        'rule_id': 'dry_hydration_completeness',
        'feature_block': 'clinical',
        'skin_types': ['건성', '약건성'],
        'direction': 'support',
        'formula_key': 'dry_hydration_completeness',
        'study_weight': 1.00,
        'directness_weight': 1.00,
        'context_weight': 1.00,
        'source': 'PMID:31532576',
        'source_title': (
            'Combined effects of glycerol and petrolatum '
            'in an emollient cream'
        ),
    },
    {
        'rule_id': 'dry_barrier_lipid_completeness',
        'feature_block': 'clinical',
        'skin_types': ['건성', '약건성'],
        'direction': 'support',
        'formula_key': 'dry_barrier_lipid_completeness',
        'study_weight': 0.80,
        'directness_weight': 0.80,
        'context_weight': 1.00,
        'source': 'PMID:8618046; PMID:9308554',
        'source_title': (
            'Physiological lipid mixtures for barrier repair'
        ),
    },
    {
        'rule_id': 'sensitive_recovery_system',
        'feature_block': 'clinical',
        'skin_types': ['민감성'],
        'direction': 'support',
        'formula_key': 'sensitive_recovery_system',
        'study_weight': 0.80,
        'directness_weight': 0.60,
        'context_weight': 1.00,
        'source': 'PMID:16209160; PMID:21982351',
        'source_title': (
            'Niacinamide/panthenol moisturizer and '
            'sensitive-skin barrier studies'
        ),
    },
    {
        'rule_id': 'sensitive_fragrance_load',
        'feature_block': 'sensory_caution',
        'skin_types': ['민감성'],
        'direction': 'caution',
        'formula_key': 'sensitive_fragrance_load',
        'study_weight': 1.00,
        'directness_weight': 0.80,
        'context_weight': 1.00,
        'source': 'SCCS fragrance allergen opinions; PMID:14572300',
        'source_title': 'Fragrance allergens and contact allergy',
    },
    {
        'rule_id': 'sensitive_acid_load',
        'feature_block': 'clinical_caution',
        'skin_types': ['민감성'],
        'direction': 'caution',
        'formula_key': 'sensitive_acid_load',
        'study_weight': 0.80,
        'directness_weight': 0.70,
        'context_weight': 0.80,
        'source': 'PMID:15725565',
        'source_title': (
            'Skin irritation and barrier effects of '
            'salicylic-acid conditions'
        ),
    },
    {
        'rule_id': 'sensitive_essential_oil_load',
        'feature_block': 'sensory_caution',
        'skin_types': ['민감성'],
        'direction': 'caution',
        'formula_key': 'sensitive_essential_oil_load',
        'study_weight': 0.60,
        'directness_weight': 0.60,
        'context_weight': 0.80,
        'source': 'PMID:24175401',
        'source_title': (
            'Botanical ingredients and allergic contact dermatitis'
        ),
    },
    {
        'rule_id': 'oily_niacinamide_prominence',
        'feature_block': 'clinical',
        'skin_types': ['지성'],
        'direction': 'support',
        'formula_key': 'oily_niacinamide_prominence',
        'study_weight': 0.80,
        'directness_weight': 1.00,
        'context_weight': 1.00,
        'source': 'PMID:16766489',
        'source_title': (
            'The effect of 2% niacinamide on facial sebum production'
        ),
    },
    {
        'rule_id': 'oily_heavy_load',
        'feature_block': 'sensory',
        'skin_types': ['지성'],
        'direction': 'exploratory',
        'formula_key': 'oily_heavy_load',
        'study_weight': 0.00,
        'directness_weight': 0.00,
        'context_weight': 0.00,
        'source': 'PMID:16116522; PMID:22085371',
        'source_title': (
            'Emollient sensory-property studies; '
            'direction learned from reviews'
        ),
    },
    {
        'rule_id': 'oily_light_system',
        'feature_block': 'sensory',
        'skin_types': ['지성'],
        'direction': 'exploratory',
        'formula_key': 'oily_light_system',
        'study_weight': 0.00,
        'directness_weight': 0.00,
        'context_weight': 0.00,
        'source': 'PMID:16116522; PMID:35512559',
        'source_title': (
            'Emollient structure, spreading and sensory properties'
        ),
    },
    {
        'rule_id': 'trouble_niacinamide_ceramide',
        'feature_block': 'clinical',
        'skin_types': ['트러블성'],
        'direction': 'support',
        'formula_key': 'trouble_niacinamide_ceramide',
        'study_weight': 1.00,
        'directness_weight': 1.00,
        'context_weight': 0.70,
        'source': 'PMID:38299457',
        'source_title': (
            'Ceramide and niacinamide moisturizer '
            'with topical acne treatment'
        ),
    },
    {
        'rule_id': 'trouble_salicylic_barrier',
        'feature_block': 'clinical',
        'skin_types': ['트러블성'],
        'direction': 'support',
        'formula_key': 'trouble_salicylic_barrier',
        'study_weight': 0.50,
        'directness_weight': 0.50,
        'context_weight': 0.80,
        'source': 'PMID:15725565; PMID:38299457',
        'source_title': (
            'Indirect compensation hypothesis: '
            'acid active with barrier support'
        ),
    },
    {
        'rule_id': 'acid_without_barrier_compensation',
        'feature_block': 'clinical_caution',
        'skin_types': ['민감성', '트러블성'],
        'direction': 'caution',
        'formula_key': 'acid_without_barrier_compensation',
        'study_weight': 0.50,
        'directness_weight': 0.50,
        'context_weight': 0.80,
        'source': 'PMID:15725565',
        'source_title': (
            'Exploratory acid-without-barrier compensation rule'
        ),
    },
]

rule_dictionary = pd.DataFrame(RULE_DEFINITIONS)

rule_dictionary['evidence_weight'] = (
    rule_dictionary['study_weight']
    * rule_dictionary['directness_weight']
    * rule_dictionary['context_weight']
).round(3)

rule_dictionary['skin_types_text'] = (
    rule_dictionary['skin_types']
    .map(lambda values: '|'.join(values))
)

display(
    rule_dictionary[
        [
            'rule_id',
            'feature_block',
            'skin_types_text',
            'direction',
            'evidence_weight',
            'source',
        ]
    ]
)


## 7. 제품×피부유형 V4 원시 Feature 생성

In [ ]:

def saturated_count(value, scale):
    value = max(float(value), 0.0)
    return 1.0 - math.exp(-value / scale)

def geometric_mean(values):
    values = [
        max(float(value), 0.0)
        for value in values
    ]

    if any(value <= 0 for value in values):
        return 0.0

    return float(
        np.prod(values) ** (1.0 / len(values))
    )

def base_rule_strength(row, formula_key):
    if formula_key == 'dry_hydration_completeness':
        return geometric_mean([
            saturated_count(
                row['v3_count_humectant'], 3
            ),
            saturated_count(
                row['v3_count_emollient'], 2
            ),
            saturated_count(
                row['v3_count_occlusive'], 2
            ),
        ])

    if formula_key == 'dry_barrier_lipid_completeness':
        return (
            row['v4_has_ceramide']
            + row['v4_has_cholesterol']
            + row['v4_has_fatty_acid']
        ) / 3.0

    if formula_key == 'sensitive_recovery_system':
        active_support = max(
            row['v4_has_panthenol'],
            row['v4_has_niacinamide'],
        )

        return geometric_mean([
            active_support,
            saturated_count(
                row['v3_count_barrier'], 2
            ),
            saturated_count(
                row['v3_count_soothing'], 2
            ),
        ])

    if formula_key == 'sensitive_fragrance_load':
        return saturated_count(
            max(
                row['v3_count_fragrance'],
                row['v4_fragrance_allergen_count'],
            ),
            1,
        )

    if formula_key == 'sensitive_acid_load':
        return saturated_count(
            max(
                row['v3_count_acid_exfoliant'],
                row['v4_acid_exact_count'],
            ),
            1,
        )

    if formula_key == 'sensitive_essential_oil_load':
        return saturated_count(
            row['v4_sensory_count_essential_oil'],
            1,
        )

    if formula_key == 'oily_niacinamide_prominence':
        return row['v4_niacinamide_prominence']

    if formula_key == 'oily_heavy_load':
        return float(np.mean([
            saturated_count(
                row['v4_sensory_count_wax_butter'], 1
            ),
            saturated_count(
                row['v4_sensory_count_plant_oil'], 2
            ),
            saturated_count(
                row['v4_sensory_count_hydrocarbon_occlusive'], 1
            ),
            saturated_count(
                row['v4_sensory_count_fatty_alcohol'], 2
            ),
        ]))

    if formula_key == 'oily_light_system':
        return float(np.mean([
            saturated_count(
                row['v4_sensory_count_volatile_silicone'], 1
            ),
            saturated_count(
                row['v4_sensory_count_light_ester'], 2
            ),
            saturated_count(
                row['v4_sensory_count_nonvolatile_silicone'], 2
            ),
        ]))

    if formula_key == 'trouble_niacinamide_ceramide':
        return float(
            row['v4_has_niacinamide']
            and row['v4_has_ceramide']
        )

    if formula_key == 'trouble_salicylic_barrier':
        return geometric_mean([
            float(row['v4_has_salicylic']),
            saturated_count(
                row['v3_count_barrier'], 2
            ),
        ])

    if formula_key == 'acid_without_barrier_compensation':
        acid_strength = saturated_count(
            max(
                row['v3_count_acid_exfoliant'],
                row['v4_acid_exact_count'],
            ),
            1,
        )

        barrier_strength = saturated_count(
            row['v3_count_barrier'],
            2,
        )

        return acid_strength * (
            1.0 - barrier_strength
        )

    raise KeyError(formula_key)

def add_rule_features(
    frame,
    rule_records,
    skin_override=None,
):
    result = frame.copy()

    support_weighted = []
    caution_weighted = []

    for rule in rule_records:
        rule_id = rule['rule_id']

        target_skins = (
            skin_override.get(
                rule_id,
                rule['skin_types'],
            )
            if skin_override is not None
            else rule['skin_types']
        )

        skin_match = (
            result['피부타입']
            .isin(target_skins)
            .astype(float)
        )

        raw_strength = result.apply(
            lambda row: base_rule_strength(
                row,
                rule['formula_key'],
            ),
            axis=1,
        )

        strength_col = (
            f'v4_rule_{rule_id}_strength'
        )
        hit_col = f'v4_rule_{rule_id}_hit'
        weighted_col = (
            f'v4_rule_{rule_id}_weighted'
        )

        result[strength_col] = (
            skin_match * raw_strength
        )
        result[hit_col] = (
            result[strength_col] > 0
        ).astype('int8')
        result[weighted_col] = (
            result[strength_col]
            * float(rule['evidence_weight'])
        )

        if (
            rule['direction'] == 'support'
            and rule['evidence_weight'] > 0
        ):
            support_weighted.append(weighted_col)

        if (
            rule['direction'] == 'caution'
            and rule['evidence_weight'] > 0
        ):
            caution_weighted.append(weighted_col)

    result['v4_evidence_support_score'] = (
        result[support_weighted].sum(axis=1)
        if support_weighted else 0.0
    )
    result['v4_evidence_caution_score'] = (
        result[caution_weighted].sum(axis=1)
        if caution_weighted else 0.0
    )
    result['v4_evidence_net_score'] = (
        result['v4_evidence_support_score']
        - result['v4_evidence_caution_score']
    )
    result['v4_evidence_conflict_score'] = (
        result['v4_evidence_support_score']
        * result['v4_evidence_caution_score']
    )

    return result

model_df = (
    target_df
    .merge(
        product_feature_df,
        left_on='상품번호',
        right_on='product_id',
        how='inner',
        validate='many_to_one',
    )
)

model_df = model_df[
    model_df['formula_group'].notna()
].copy()

model_df = add_rule_features(
    model_df,
    rule_dictionary.to_dict(orient='records'),
)

print("제품×피부유형 행:", len(model_df))
print("연결 제품 수:", model_df['product_id'].nunique())
print("처방 그룹 수:", model_df['formula_group'].nunique())
display(model_df.head())



## 8. Fold 내부 empirical-Bayes posterior

문헌 prior는 각 규칙의 기대 방향을 약하게 반영하고, 실제 Train 리뷰 관찰값으로 보정합니다.

- Support 규칙: prior가 피부유형 baseline보다 약간 낮은 위험률
- Caution 규칙: prior가 baseline보다 약간 높은 위험률
- 실제 Train 데이터가 충분하면 관찰값의 영향이 커짐
- Outer Train은 Inner Group CV로 cross-fitting
- Outer Test는 Outer Train 전체로 계산

따라서 Test Target은 posterior 계산에 사용되지 않습니다.


In [ ]:

def safe_logit(probability, eps=1e-5):
    probability = float(
        np.clip(probability, eps, 1.0 - eps)
    )
    return math.log(
        probability / (1.0 - probability)
    )

def sigmoid(value):
    return 1.0 / (1.0 + math.exp(-value))

def posterior_parameters(
    fit_frame,
    rule,
    target_column='high_risk',
):
    target_skins = rule['skin_types']

    baseline_frame = fit_frame[
        fit_frame['피부타입'].isin(target_skins)
    ]

    if baseline_frame.empty:
        baseline_frame = fit_frame

    baseline_rate = (
        baseline_frame[target_column].mean()
    )

    strength_col = (
        f"v4_rule_{rule['rule_id']}_strength"
    )

    hit_frame = fit_frame[
        fit_frame[strength_col] > 0
    ]

    n_hit = len(hit_frame)
    k_hit = float(
        hit_frame[target_column].sum()
    )

    evidence_weight = float(
        rule['evidence_weight']
    )

    prior_strength = (
        2.0 + 8.0 * evidence_weight
    )

    if rule['direction'] == 'support':
        direction_sign = -1.0
    elif rule['direction'] == 'caution':
        direction_sign = 1.0
    else:
        direction_sign = 0.0

    prior_shift = (
        direction_sign
        * 0.25
        * evidence_weight
    )

    prior_mean = sigmoid(
        safe_logit(baseline_rate)
        + prior_shift
    )

    posterior_mean = (
        k_hit
        + prior_strength * prior_mean
    ) / (
        n_hit + prior_strength
    )

    posterior_delta = (
        safe_logit(posterior_mean)
        - safe_logit(baseline_rate)
    )

    reliability = (
        n_hit / (n_hit + prior_strength)
        if n_hit + prior_strength > 0
        else 0.0
    )

    return {
        'baseline_rate': baseline_rate,
        'prior_mean': prior_mean,
        'posterior_mean': posterior_mean,
        'posterior_delta': posterior_delta,
        'reliability': reliability,
        'n_hit': n_hit,
    }

def apply_posterior_parameters(
    frame,
    rule,
    parameters,
):
    strength_col = (
        f"v4_rule_{rule['rule_id']}_strength"
    )

    hit = (
        frame[strength_col] > 0
    ).astype(float)

    strength = frame[strength_col].astype(float)

    return (
        hit
        * strength
        * parameters['posterior_delta']
        * parameters['reliability']
    )

def add_cross_fitted_posterior(
    train_frame,
    test_frame,
    rules,
    target_column='high_risk',
):
    train_result = train_frame.copy()
    test_result = test_frame.copy()

    posterior_columns = [
        f"v4_post_{rule['rule_id']}"
        for rule in rules
        if rule['evidence_weight'] > 0
    ]

    for column in posterior_columns:
        train_result[column] = 0.0
        test_result[column] = 0.0

    inner_splitter = StratifiedGroupKFold(
        n_splits=INNER_POSTERIOR_SPLITS,
        shuffle=True,
        random_state=RANDOM_SEED,
    )

    for inner_fit_index, inner_valid_index in (
        inner_splitter.split(
            train_result,
            train_result[target_column],
            train_result['formula_group'],
        )
    ):
        inner_fit = train_result.iloc[
            inner_fit_index
        ]
        inner_valid = train_result.iloc[
            inner_valid_index
        ]

        for rule in rules:
            if rule['evidence_weight'] <= 0:
                continue

            parameters = posterior_parameters(
                inner_fit,
                rule,
                target_column,
            )

            column = f"v4_post_{rule['rule_id']}"

            train_result.loc[
                train_result.index[inner_valid_index],
                column,
            ] = apply_posterior_parameters(
                inner_valid,
                rule,
                parameters,
            ).to_numpy()

    # Test는 Outer Train 전체에서 추정
    for rule in rules:
        if rule['evidence_weight'] <= 0:
            continue

        parameters = posterior_parameters(
            train_result,
            rule,
            target_column,
        )

        column = f"v4_post_{rule['rule_id']}"

        test_result[column] = (
            apply_posterior_parameters(
                test_result,
                rule,
                parameters,
            )
        )

    for frame in [train_result, test_result]:
        frame['v4_posterior_positive'] = (
            frame[posterior_columns]
            .clip(lower=0)
            .sum(axis=1)
        )
        frame['v4_posterior_negative'] = (
            -frame[posterior_columns]
            .clip(upper=0)
            .sum(axis=1)
        )
        frame['v4_posterior_net'] = (
            frame[posterior_columns]
            .sum(axis=1)
        )
        frame['v4_posterior_abs'] = (
            frame[posterior_columns]
            .abs()
            .sum(axis=1)
        )

    return (
        train_result,
        test_result,
        posterior_columns,
    )


## 9. Ablation 실험 설정

In [ ]:

skin_dummy_df = pd.get_dummies(
    model_df['피부타입'],
    prefix='skin',
    dtype='int8',
)

experiment_df = pd.concat(
    [
        model_df.reset_index(drop=True),
        skin_dummy_df.reset_index(drop=True),
    ],
    axis=1,
)

skin_columns = list(skin_dummy_df.columns)

v3_count_columns = [
    column
    for column in experiment_df.columns
    if column.startswith('v3_count_')
]

sensory_base_columns = [
    column
    for column in experiment_df.columns
    if column.startswith('v4_sensory_count_')
]

order_proxy_columns = [
    'v4_niacinamide_top10',
    'v4_niacinamide_prominence',
    'v4_panthenol_top10',
    'v4_panthenol_prominence',
    'v4_ceramide_prominence',
    'v4_salicylic_prominence',
]

clinical_rule_ids = set(
    rule_dictionary.loc[
        rule_dictionary['feature_block'].isin([
            'clinical',
            'clinical_caution',
        ]),
        'rule_id',
    ]
)

sensory_rule_ids = set(
    rule_dictionary.loc[
        rule_dictionary['feature_block'].isin([
            'sensory',
            'sensory_caution',
        ]),
        'rule_id',
    ]
)

clinical_rule_columns = [
    f'v4_rule_{rule_id}_strength'
    for rule_id in clinical_rule_ids
]

sensory_rule_columns = [
    f'v4_rule_{rule_id}_strength'
    for rule_id in sensory_rule_ids
]

evidence_score_columns = [
    'v4_evidence_support_score',
    'v4_evidence_caution_score',
    'v4_evidence_net_score',
    'v4_evidence_conflict_score',
]

raw_v4_columns = (
    clinical_rule_columns
    + sensory_base_columns
    + sensory_rule_columns
    + order_proxy_columns
    + evidence_score_columns
)

BASE_FEATURE_SETS = {
    'M0_skin_only': skin_columns,
    'M1_skin_v3_count': (
        skin_columns
        + v3_count_columns
    ),
    'M2_v3_clinical_rules': (
        skin_columns
        + v3_count_columns
        + clinical_rule_columns
    ),
    'M3_v3_sensory': (
        skin_columns
        + v3_count_columns
        + sensory_base_columns
        + sensory_rule_columns
    ),
    'M4_v3_order_proxy': (
        skin_columns
        + v3_count_columns
        + order_proxy_columns
    ),
    'M5_v3_evidence_scores': (
        skin_columns
        + v3_count_columns
        + evidence_score_columns
    ),
    'M6_v3_v4_raw_all': (
        skin_columns
        + v3_count_columns
        + raw_v4_columns
    ),
}

MODEL_NAMES = []

if RUN_CATBOOST:
    MODEL_NAMES.append('catboost')

if RUN_ELASTICNET:
    MODEL_NAMES.append('elasticnet')

print("모델:", MODEL_NAMES)
print("V3 Count:", len(v3_count_columns))
print("Raw V4:", len(set(raw_v4_columns)))


## 10. Group CV 평가 실행

In [ ]:

def top_quartile_metrics(y_true, probabilities):
    n_top = max(
        1,
        int(math.ceil(len(y_true) * 0.25)),
    )

    top_indices = np.argsort(
        -probabilities
    )[:n_top]

    positive_total = max(
        int(y_true.sum()),
        1,
    )

    recall_top25 = (
        y_true[top_indices].sum()
        / positive_total
    )

    baseline_rate = y_true.mean()
    top_rate = y_true[top_indices].mean()

    lift_top25 = (
        top_rate / baseline_rate
        if baseline_rate > 0
        else np.nan
    )

    return recall_top25, lift_top25

def build_model(model_name, seed):
    if model_name == 'catboost':
        return CatBoostClassifier(
            iterations=180,
            depth=4,
            learning_rate=0.03,
            loss_function='Logloss',
            eval_metric='AUC',
            auto_class_weights='Balanced',
            l2_leaf_reg=6,
            random_seed=seed,
            verbose=False,
            allow_writing_files=False,
        )

    if model_name == 'elasticnet':
        return make_pipeline(
            StandardScaler(),
            LogisticRegression(
                penalty='elasticnet',
                solver='saga',
                l1_ratio=0.5,
                C=0.5,
                class_weight='balanced',
                max_iter=5000,
                random_state=seed,
            ),
        )

    raise ValueError(model_name)

def fit_model(
    model,
    model_name,
    x_train,
    y_train,
    sample_weight,
):
    if model_name == 'catboost':
        model.fit(
            x_train,
            y_train,
            sample_weight=sample_weight,
        )
    else:
        model.fit(
            x_train,
            y_train,
            logisticregression__sample_weight=(
                sample_weight
            ),
        )

def evaluate_all():
    y = experiment_df['high_risk'].to_numpy()
    groups = experiment_df['formula_group'].to_numpy()

    if USE_REVIEW_COUNT_WEIGHT:
        sample_weights = np.sqrt(
            experiment_df['review_count']
            .clip(upper=MAX_REVIEW_WEIGHT_COUNT)
            .to_numpy()
        )
    else:
        sample_weights = np.ones(len(experiment_df))

    fold_rows = []

    for repeat in range(N_REPEATS):
        seed = RANDOM_SEED + repeat

        splitter = StratifiedGroupKFold(
            n_splits=N_SPLITS,
            shuffle=True,
            random_state=seed,
        )

        for fold, (
            train_index,
            test_index,
        ) in enumerate(
            splitter.split(
                experiment_df,
                y,
                groups,
            )
        ):
            outer_train = experiment_df.iloc[
                train_index
            ].copy()
            outer_test = experiment_df.iloc[
                test_index
            ].copy()

            posterior_train, posterior_test, (
                posterior_rule_columns
            ) = add_cross_fitted_posterior(
                outer_train,
                outer_test,
                rule_dictionary.to_dict(
                    orient='records'
                ),
            )

            posterior_aggregate_columns = [
                'v4_posterior_positive',
                'v4_posterior_negative',
                'v4_posterior_net',
                'v4_posterior_abs',
            ]

            fold_feature_sets = dict(
                BASE_FEATURE_SETS
            )

            fold_feature_sets[
                'M7_v3_v4_posterior'
            ] = (
                BASE_FEATURE_SETS[
                    'M6_v3_v4_raw_all'
                ]
                + posterior_rule_columns
                + posterior_aggregate_columns
            )

            for model_name in MODEL_NAMES:
                for feature_set_name, columns in (
                    fold_feature_sets.items()
                ):
                    model = build_model(
                        model_name,
                        seed + fold,
                    )

                    fit_model(
                        model,
                        model_name,
                        posterior_train[columns],
                        posterior_train['high_risk'],
                        sample_weights[train_index],
                    )

                    probabilities = (
                        model.predict_proba(
                            posterior_test[columns]
                        )[:, 1]
                    )

                    y_test = posterior_test[
                        'high_risk'
                    ].to_numpy()

                    recall_top25, lift_top25 = (
                        top_quartile_metrics(
                            y_test,
                            probabilities,
                        )
                    )

                    fold_rows.append({
                        'section': 'cv_fold',
                        'model': model_name,
                        'feature_set': feature_set_name,
                        'repeat': repeat,
                        'fold': fold,
                        'roc_auc': roc_auc_score(
                            y_test,
                            probabilities,
                        ),
                        'pr_auc': average_precision_score(
                            y_test,
                            probabilities,
                        ),
                        'recall_top25': recall_top25,
                        'lift_top25': lift_top25,
                        'n_train': len(train_index),
                        'n_test': len(test_index),
                    })

    return pd.DataFrame(fold_rows)

cv_fold_results = evaluate_all()

cv_summary = (
    cv_fold_results
    .groupby(
        ['model', 'feature_set'],
        as_index=False,
    )
    .agg(
        roc_auc_mean=('roc_auc', 'mean'),
        roc_auc_std=('roc_auc', 'std'),
        pr_auc_mean=('pr_auc', 'mean'),
        pr_auc_std=('pr_auc', 'std'),
        recall_top25_mean=(
            'recall_top25',
            'mean',
        ),
        lift_top25_mean=(
            'lift_top25',
            'mean',
        ),
        n_evaluations=('pr_auc', 'size'),
    )
)

cv_summary.insert(
    0,
    'section',
    'cv_summary',
)

display(
    cv_summary.sort_values(
        ['model', 'pr_auc_mean'],
        ascending=[True, False],
    )
)


## 11. 규칙별 리뷰 방향 진단

In [ ]:

diagnostic_rows = []

for rule in rule_dictionary.to_dict(
    orient='records'
):
    rule_id = rule['rule_id']
    hit_col = f'v4_rule_{rule_id}_hit'
    strength_col = (
        f'v4_rule_{rule_id}_strength'
    )

    active = model_df[hit_col] == 1
    inactive = model_df[hit_col] == 0

    diagnostic_rows.append({
        'section': 'rule_diagnostic',
        'rule_id': rule_id,
        'direction_expected': rule['direction'],
        'evidence_weight': rule['evidence_weight'],
        'active_rows': int(active.sum()),
        'active_products': int(
            model_df.loc[
                active,
                'product_id',
            ].nunique()
        ),
        'active_high_risk_rate': (
            model_df.loc[
                active,
                'high_risk',
            ].mean()
            if active.sum() else np.nan
        ),
        'inactive_high_risk_rate': (
            model_df.loc[
                inactive,
                'high_risk',
            ].mean()
            if inactive.sum() else np.nan
        ),
        'active_negative_rate_mean': (
            model_df.loc[
                active,
                'negative_rate',
            ].mean()
            if active.sum() else np.nan
        ),
        'inactive_negative_rate_mean': (
            model_df.loc[
                inactive,
                'negative_rate',
            ].mean()
            if inactive.sum() else np.nan
        ),
        'mean_strength': (
            model_df.loc[
                active,
                strength_col,
            ].mean()
            if active.sum() else np.nan
        ),
    })

rule_diagnostics = pd.DataFrame(
    diagnostic_rows
)

display(rule_diagnostics)



## 12. 무작위 피부유형 규칙 대조군

문헌 규칙과 같은 성분 조건·가중치를 유지하고 대상 피부유형만 무작위로 변경합니다.

문헌 규칙이 무작위 규칙보다 안정적으로 좋아야 단순 Feature 증가 효과가 아니라
도메인 지식 선택의 효과라고 주장할 수 있습니다.


In [ ]:

random_control_rows = []

def random_skin_override(
    rules,
    skin_types,
    rng,
):
    override = {}

    for rule in rules:
        n_skin = len(rule['skin_types'])

        selected = list(
            rng.choice(
                skin_types,
                size=n_skin,
                replace=False,
            )
        )

        override[rule['rule_id']] = selected

    return override

if RUN_RANDOM_RULE_CONTROL:
    fixed_splitter = StratifiedGroupKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_SEED,
    )

    fixed_folds = list(
        fixed_splitter.split(
            experiment_df,
            experiment_df['high_risk'],
            experiment_df['formula_group'],
        )
    )

    observed_columns = (
        skin_columns
        + v3_count_columns
        + raw_v4_columns
    )

    def fixed_elasticnet_pr_auc(
        data,
        columns,
    ):
        scores = []

        for fold, (
            train_index,
            test_index,
        ) in enumerate(fixed_folds):
            model = build_model(
                'elasticnet',
                RANDOM_SEED + fold,
            )

            if USE_REVIEW_COUNT_WEIGHT:
                weights = np.sqrt(
                    data.iloc[
                        train_index
                    ]['review_count']
                    .clip(
                        upper=MAX_REVIEW_WEIGHT_COUNT
                    )
                    .to_numpy()
                )
            else:
                weights = np.ones(
                    len(train_index)
                )

            fit_model(
                model,
                'elasticnet',
                data.iloc[
                    train_index
                ][columns],
                data.iloc[
                    train_index
                ]['high_risk'],
                weights,
            )

            probabilities = (
                model.predict_proba(
                    data.iloc[
                        test_index
                    ][columns]
                )[:, 1]
            )

            scores.append(
                average_precision_score(
                    data.iloc[
                        test_index
                    ]['high_risk'],
                    probabilities,
                )
            )

        return float(np.mean(scores))

    observed_pr_auc = fixed_elasticnet_pr_auc(
        experiment_df,
        observed_columns,
    )

    skin_types = sorted(
        experiment_df['피부타입'].unique()
    )

    for random_index in range(
        N_RANDOM_RULE_SETS
    ):
        rng = np.random.default_rng(
            RANDOM_SEED + 1000 + random_index
        )

        override = random_skin_override(
            rule_dictionary.to_dict(
                orient='records'
            ),
            skin_types,
            rng,
        )

        random_rule_frame = add_rule_features(
            model_df,
            rule_dictionary.to_dict(
                orient='records'
            ),
            skin_override=override,
        )

        random_skin_dummies = pd.get_dummies(
            random_rule_frame['피부타입'],
            prefix='skin',
            dtype='int8',
        )

        random_experiment = pd.concat(
            [
                random_rule_frame.reset_index(
                    drop=True
                ),
                random_skin_dummies.reset_index(
                    drop=True
                ),
            ],
            axis=1,
        )

        random_rule_columns = [
            column
            for column in random_experiment.columns
            if column.startswith('v4_rule_')
            and column.endswith('_strength')
        ]

        random_columns = (
            skin_columns
            + v3_count_columns
            + sensory_base_columns
            + order_proxy_columns
            + random_rule_columns
            + evidence_score_columns
        )

        random_pr_auc = (
            fixed_elasticnet_pr_auc(
                random_experiment,
                random_columns,
            )
        )

        random_control_rows.append({
            'section': 'random_rule_control',
            'random_index': random_index,
            'observed_pr_auc': observed_pr_auc,
            'random_pr_auc': random_pr_auc,
            'delta': (
                observed_pr_auc
                - random_pr_auc
            ),
            'skin_override_json': json.dumps(
                override,
                ensure_ascii=False,
                sort_keys=True,
            ),
        })

    random_control_df = pd.DataFrame(
        random_control_rows
    )

    empirical_p = (
        1
        + (
            random_control_df[
                'random_pr_auc'
            ] >= observed_pr_auc
        ).sum()
    ) / (
        1 + len(random_control_df)
    )

    random_summary_df = pd.DataFrame([{
        'section': 'random_rule_summary',
        'observed_pr_auc': observed_pr_auc,
        'random_pr_auc_mean': (
            random_control_df[
                'random_pr_auc'
            ].mean()
        ),
        'random_pr_auc_std': (
            random_control_df[
                'random_pr_auc'
            ].std()
        ),
        'observed_percentile': (
            random_control_df[
                'random_pr_auc'
            ].lt(observed_pr_auc).mean()
        ),
        'empirical_p_value': empirical_p,
        'n_random_sets': len(
            random_control_df
        ),
    }])

    display(random_summary_df)
else:
    random_control_df = pd.DataFrame()
    random_summary_df = pd.DataFrame()
    print("무작위 규칙 대조를 건너뜁니다.")


## 13. 최종 CSV 저장

In [ ]:

FEATURE_OUTPUT_PATH = (
    OUTPUT_DIR / 'v4_model_features.csv'
)
RULE_OUTPUT_PATH = (
    OUTPUT_DIR / 'v4_rule_dictionary.csv'
)
EXPERIMENT_OUTPUT_PATH = (
    OUTPUT_DIR / 'v4_experiment_results.csv'
)

# 일반 모델 입력 파일에는 Target 누수가 없는 Raw V4만 저장합니다.
# Posterior는 split마다 다시 계산해야 하므로 이 파일에 고정 저장하지 않습니다.
feature_columns = [
    'product_id',
    'product_name',
    '피부타입',
    'review_count',
    'negative_count',
    'negative_rate',
    'high_risk',
    'formula_group',
] + v3_count_columns + sorted(set(raw_v4_columns))

feature_output = model_df[
    feature_columns
].copy()

rule_output = rule_dictionary.drop(
    columns=['skin_types']
).copy()

experiment_parts = [
    cv_summary,
    cv_fold_results,
    rule_diagnostics,
]

if not random_summary_df.empty:
    experiment_parts.append(
        random_summary_df
    )

if not random_control_df.empty:
    experiment_parts.append(
        random_control_df
    )

experiment_output = pd.concat(
    experiment_parts,
    ignore_index=True,
    sort=False,
)

feature_output.to_csv(
    FEATURE_OUTPUT_PATH,
    index=False,
    encoding='utf-8-sig',
)
rule_output.to_csv(
    RULE_OUTPUT_PATH,
    index=False,
    encoding='utf-8-sig',
)
experiment_output.to_csv(
    EXPERIMENT_OUTPUT_PATH,
    index=False,
    encoding='utf-8-sig',
)

assert feature_output[
    ['product_id', '피부타입']
].duplicated().sum() == 0
assert feature_output.isna().sum().sum() == 0
assert feature_output['high_risk'].isin(
    [0, 1]
).all()

print("저장 완료")
print("1.", FEATURE_OUTPUT_PATH)
print("2.", RULE_OUTPUT_PATH)
print("3.", EXPERIMENT_OUTPUT_PATH)



## 14. 최종 채택 기준

V4는 다음 조건을 만족할 때만 최종 모델에 채택합니다.

1. `M1_skin_v3_count`보다 PR-AUC 또는 Lift@25% 개선
2. 최소 리뷰 수 5개와 10개 조건에서 개선 방향 유지
3. 동일 처방 Group CV에서 개선 유지
4. 문헌 규칙이 무작위 피부유형 규칙보다 우수
5. Posterior 모델이 Raw V4보다 안정적이거나 개선
6. 특정 소수 브랜드·제품군만 식별하는 규칙이 아님

추천 최종 비교:

```text
M1: Skin + V3 Count
M2: M1 + 임상 조합
M3: M1 + 사용감 조합
M4: M1 + 성분 순서 Proxy
M5: M1 + 문헌 집계점수
M6: M1 + Raw V4 전체
M7: M6 + Fold 내부 posterior
```

문헌 기대 방향과 실제 리뷰 방향이 반대여도 규칙을 임의 삭제하지 않습니다.
불일치 자체를 `문헌 근거와 소비자 리뷰의 차이`로 분석합니다.
